In [1]:
from pysat.formula import CNF, IDPool
from pysat.solvers import Solver, Minisat22
from pysat.card import CardEnc, EncType

In [2]:
# generate CNF encoding given length and specific lag
def generate_encoding_lag(r, lag, cnf, pool, varmap, pmap_lags, seqs):
    # add variable to map
    def seq_id(seq, i):
        key = f'{seq}_{i}'
        if key not in varmap:
            varmap[key] = pool.id(key)
        return varmap[key]

    # create auxiliary variable for x * y
    def add_reified_xnor(p, x, y):
        cnf.append([-p, -x,  y])
        cnf.append([-p,  x, -y])
        cnf.append([ p, -x, -y])
        cnf.append([ p,  x,  y])

    # for each sequence, add clause for new combination
    pmap = {}
    p_lits = []
    for seq in seqs:
        for i in range(r - lag):
            j = i + lag
            seq_i = seq_id(seq, i)
            seq_j = seq_id(seq, j)
            p_key = f'p_{seq}_{i}_{j}'
            p_id = pool.id(p_key)
            pmap[p_key] = p_id
            p_lits.append(p_id)
            add_reified_xnor(p_id, seq_i, seq_j)

    # verify sum is 0
    eq = CardEnc.equals(lits=p_lits, bound=2 * (r - lag), encoding=EncType.totalizer, vpool=pool)
    cnf.extend(eq.clauses)
    pmap_lags[lag] = pmap

In [3]:
# generate CNF encoding given length
def generate_encoding(r):
    cnf = CNF()
    pool = IDPool()
    varmap = {}
    pmap_lags = {}
    seqs = ['X', 'Y', 'Z', 'W']

    for lag in range(1, r):
        generate_encoding_lag(r, lag, cnf, pool, varmap, pmap_lags, seqs)
        
    return cnf, varmap, pmap_lags, pool

In [4]:
cnf, varmap, pmap_lags, pool = generate_encoding(3)
print(varmap)
print(cnf.clauses)

{'X_0': 1, 'X_1': 2, 'X_2': 4, 'Y_0': 6, 'Y_1': 7, 'Y_2': 9, 'Z_0': 11, 'Z_1': 12, 'Z_2': 14, 'W_0': 16, 'W_1': 17, 'W_2': 19}
[[-3, -1, 2], [-3, 1, -2], [3, -1, -2], [3, 1, 2], [-5, -2, 4], [-5, 2, -4], [5, -2, -4], [5, 2, 4], [-8, -6, 7], [-8, 6, -7], [8, -6, -7], [8, 6, 7], [-10, -7, 9], [-10, 7, -9], [10, -7, -9], [10, 7, 9], [-13, -11, 12], [-13, 11, -12], [13, -11, -12], [13, 11, 12], [-15, -12, 14], [-15, 12, -14], [15, -12, -14], [15, 12, 14], [-18, -16, 17], [-18, 16, -17], [18, -16, -17], [18, 16, 17], [-20, -17, 19], [-20, 17, -19], [20, -17, -19], [20, 17, 19], [-33, 21], [-34, 22], [-35, 23], [-36, 24], [-29, 21], [-30, 22], [-31, 23], [-32, 24], [-29, -33, 22], [-29, -34, 23], [-29, -35, 24], [-29, -36, 25], [-30, -33, 23], [-30, -34, 24], [-30, -35, 25], [-30, -36, 26], [-31, -33, 24], [-31, -34, 25], [-31, -35, 26], [-31, -36, 27], [-32, -33, 25], [-32, -34, 26], [-32, -35, 27], [-32, -36, 28], [-39, 33], [-40, 34], [-37, 33], [-38, 34], [-37, -39, 34], [-37, -40, 35], 

In [5]:
def make_assumptions(varmap, assign):
    lits = []
    for (seq,i), val in assign.items():
        v = varmap[f'{seq}_{i}']
        lits.append(v if val == 1 else -v)
    return lits

def verify_assignment(assignment, r, varmap, cnf):
    print(assignment)
    assign = {}
    seqs = ['X', 'Y', 'Z', 'W']

    seq_num = 0
    for seq in seqs: 
        for i in range(r):
            assign[(seq, i)] = assignment[seq_num][i]
        seq_num += 1

    with Minisat22(bootstrap_with=cnf) as solver:
        print('formula is', f'{"s" if solver.solve(assumptions=make_assumptions(varmap, assign)) else "uns"}atisfiable')

In [32]:
from sequence_generator import generate_all_turyns, generate_invalid_turyn

r = 3
turyn_seqs = generate_invalid_turyn(r)
cnf, varmap, pmap_lags, pool = generate_encoding(r)
verify_assignment(turyn_seqs, r, varmap, cnf)

[(1, -1, -1), (1, 1, 1), (1, 1, -1), (1, -1, -1)]
formula is unsatisfiable
